# 基线 vs 增强方法对比

运行小规模流水线，比较只使用原始电压相关的基线与默认增强评分在相同真值下的 F1。

In [1]:
from pathlib import Path


def repo_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "configs" / "default.yaml").exists():
            return candidate
    raise FileNotFoundError("未找到仓库根目录")

ROOT = repo_root()

In [2]:
from ltverify.config import load_config
from ltverify.io import read_table
from ltverify.pipeline import run_pipeline

run_dir = run_pipeline(ROOT / "tests/fixtures/small_config.yaml")
cfg = load_config(ROOT / "tests/fixtures/small_config.yaml")
features = read_table(run_dir / "candidate_features.parquet")
ledger = read_table(run_dir / "reported_ledger.csv")
truth = read_table(run_dir / "truth_topology.csv")
enhanced_predictions = read_table(run_dir / "predictions.parquet")
print("运行目录:", run_dir.resolve().relative_to(ROOT).as_posix())

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


运行目录:

notebooks/runs/run-20260816T155238-dca6e539-a76c64

In [3]:
from ltverify.evaluation import evaluate_predictions
from ltverify.scoring import ScoreWeights, diagnose, score_candidates

baseline_weights = ScoreWeights(raw_corr=1.0, residual_corr=0.0, diff_corr=0.0, rolling_corr_median=0.0, rolling_corr_q10=0.0, event_match=0.0, active_power_corr=0.0)
baseline_scored = score_candidates(features, baseline_weights)
baseline_predictions = diagnose(baseline_scored, ledger, cfg.scoring)
enhanced_result = evaluate_predictions(enhanced_predictions, truth, ledger, features)
baseline_result = evaluate_predictions(baseline_predictions, truth, ledger, baseline_scored)
print("增强 F1:", round(enhanced_result.metrics["f1"], 3))
print("基线 F1:", round(baseline_result.metrics["f1"], 3))

增强 F1:

0.0

基线 F1:

0.0

In [4]:
import plotly.graph_objects as go

methods = ["基线", "增强"]
f1_values = [baseline_result.metrics["f1"], enhanced_result.metrics["f1"]]
figure = go.Figure(go.Bar(x=methods, y=f1_values, text=[round(v, 3) for v in f1_values], textposition="auto"))
figure.update_layout(title="02 基线 vs 增强 F1（小规模冒烟场景）", yaxis_title="F1")
output = ROOT / "reports/figures/02_baseline_analysis.html"
output.parent.mkdir(parents=True, exist_ok=True)
figure.write_html(output)
print("图已保存:", output.resolve().relative_to(ROOT).as_posix())

图已保存:

reports/figures/02_baseline_analysis.html